# Bridge data: one month of ERA5-Land and CERRA-Land on the OPERA grid

S7 (#13). ERA5-Land (hourly, 0.1°, about 9 km) and CERRA-Land (daily, 5.5 km, with rain gauges) are the
pan-European bridge candidates for the ODYSSEY → CIRRUS level correction (H3). They are downloaded over
Belgium + 350 km by `scripts/fetch_cds.py` (`docs/data.md`, *Derived datasets*). This notebook reads one month
of both and sets it beside ODYSSEY on the OPERA grid, as coarse cells: the bridge is never interpolated to
pixels, and ODYSSEY is averaged over the pixels of each bridge cell (`opera_ds/bridge.py`).

**Month.** July 2013: summer rain, and a month already downloaded for both datasets (CERRA-Land is still being
downloaded, newest year first, in S8). ODYSSEY has 2,969 of its 2,976 frames (`docs/data.md`, *ODYSSEY before
April 2013 is sparse*, lists the gaps of 2013).

**Window.** A CERRA-Land day runs from 06 UTC of the previous day to 06 UTC (data user guide, in `docs/data.md`),
so all three are summed over 1/7/2013 06 UTC → 1/8/2013 06 UTC:

- CERRA-Land: the 31 days labelled 2/7 … 1/8 at 06 UTC.
- ERA5-Land: the 744 hours ending 07 UTC 1/7 … 06 UTC 1/8, from the accumulation from 00 UTC (`docs/data.md`).
- ODYSSEY: the frames labelled 06:15 1/7 … 06:00 1/8. A frame labelled NT covers [NT−10, NT+5] (`docs/data.md`,
  *Time windows*), so the frames cover 06:05 → 06:05, 5 min later than the bridge.

**ODYSSEY rain.** Marshall-Palmer from 7 dBZ, capped at 55 dBZ, after the first-version cleaning of H2
(`opera_ds.clean`): echoes above 70 dBZ, Gabella, and the static mask. Here the static mask is counted on the
composites of this same month and every masked pixel is always in use; H2 counts it on past composites and
switches it with `how/nodes`. Missing frames are filled with the pixel's mean rate of the month; pixels valid
in less than 90% of the frames are left out.

**What to expect.** ODYSSEY is a maximum-reflectivity composite, and its Marshall-Palmer sums are an index,
not rain: they come out far above the rain that falls (`docs/method.md`, *Catalogue: first run on ODYSSEY
2023 and CIRRUS 2025*). The level is what H3 corrects; this notebook only shows the data side by side, with
the level gap and how well the spatial patterns agree.

In [ ]:
%load_ext autoreload
%autoreload 2
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import datetime as dt
import sys
from pathlib import Path

import warnings

import matplotlib.pyplot as plt
import numpy as np
from pyproj import Proj
from scipy.stats import spearmanr
from shapely import transform

from opera_ds.bridge import cell_index, cell_mean, cerraland_sum, era5land_sum, monthly_files, pixel_lonlat
from opera_ds.clean import StaticCounts, StaticMask, clean_composite, rain_rate
from opera_ds.odim_io import index_files, read_odim_composite

sys.path.insert(0, str(Path.cwd().parent))          # scripts/ is not a package of opera_ds
from scripts.make_region_mask import country_geometry, grid_mask

## Parameters

In [ ]:
START, END = dt.datetime(2013, 7, 1, 6), dt.datetime(2013, 8, 1, 6)
MONTHS = [(2013, 7), (2013, 8)]                     # August holds the last day of the window
ODYSSEY_GLOB = str(DATA_DIR / "odyssey_bel" / "2013" / "0[78]" / "*@DBZH_QIND.h5")
MIN_VALID = 0.9                                     # share of frames a pixel needs
OUT = RESULTS_DIR / "bridge"
OUT.mkdir(parents=True, exist_ok=True)

## 1. ERA5-Land and CERRA-Land over the window

The readers check that every hour (ERA5-Land) and every day (CERRA-Land) of the window is there.

In [ ]:
era, era_lat, era_lon = era5land_sum(monthly_files(DATA_DIR / "era5land_bel", "era5-land", MONTHS), START, END)
cer, cer_lat, cer_lon = cerraland_sum(monthly_files(DATA_DIR / "cerraland_bel", "cerra-land", MONTHS), START, END)
print(f"ERA5-Land  {era.shape} cells, {era_lat.min():.1f}–{era_lat.max():.1f} N, {era_lon.min():.1f}–{era_lon.max():.1f} E,"
      f" domain mean {np.nanmean(era):.1f} mm")
print(f"CERRA-Land {cer.shape} cells, {cer_lat.min():.1f}–{cer_lat.max():.1f} N, {cer_lon.min():.1f}–{cer_lon.max():.1f} E,"
      f" domain mean {np.nanmean(cer):.1f} mm")

## 2. ODYSSEY over the same window

In [ ]:
idx = index_files(ODYSSEY_GLOB, r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")
labels = [START + dt.timedelta(minutes=15 * k) for k in range(1, int((END - START).total_seconds() // 900) + 1)]
present = [t for t in labels if t in idx]
print(f"{len(present)} of {len(labels)} frames; missing: {[f'{t:%d/%m %H:%M}' for t in labels if t not in idx]}")

# pass 1: counts for the static mask, on this month's composites
counts = None
for t in present:
    fr = read_odim_composite(idx[t])
    if counts is None:
        counts, where = StaticCounts.zeros(fr.values.shape), fr.where
    counts.add(fr.to_field(0.0))
static = StaticMask(counts.mask(), pixel_km=2.0)
print(f"static mask: {static.mask.sum()} pixels")

# pass 2: cleaned rain, summed
acc, cnt = np.zeros(static.mask.shape), np.zeros(static.mask.shape, int)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)          # wradlib's Gabella on NaN outside the coverage
    for t in present:
        fr = read_odim_composite(idx[t])
        r = rain_rate(clean_composite(fr.to_field(0.0), static)) * 0.25          # mm in 15 min
        acc += np.where(fr.coverage, r, 0.0)
        cnt += fr.coverage
share = cnt / len(labels)
ody = np.where(share >= MIN_VALID, acc / np.maximum(cnt, 1) * len(labels), np.nan)
print(f"grid {ody.shape} at 2 km; pixels kept {np.isfinite(ody).mean():.1%}; domain mean {np.nanmean(ody):.1f} mm")

## 3. On the OPERA grid as coarse cells

Each ODYSSEY pixel takes the bridge cell with the nearest centre. The bridge value is copied to all its pixels
(blocks in the maps), and ODYSSEY is averaged over the pixels of each cell; a cell counts only if all its pixels
are valid. Belgium is the Natural Earth polygon, pixel centres inside.

In [ ]:
lon, lat = pixel_lonlat(where)
ie = cell_index(era_lat, era_lon, lat, lon)
ic = cell_index(cer_lat, cer_lon, lat, lon)
ody_e, _ = cell_mean(ody, ie, era.size)
ody_c, _ = cell_mean(ody, ic, cer.size)

bel = grid_mask(where, country_geometry("BEL", str(DATA_DIR / "naturalearth")))
cells_e = np.unique(ie[bel])
cells_c = np.unique(ic[bel])
print(f"Belgium: {bel.sum()} ODYSSEY pixels, {cells_e.size} ERA5-Land cells, {cells_c.size} CERRA-Land cells")
print(f"ERA5-Land cell of about {bel.sum() / cells_e.size:.0f} pixels at 2 km, CERRA-Land of about {bel.sum() / cells_c.size:.0f}")

In [ ]:
rows = [("ERA5-Land", era, ody_e, cells_e), ("CERRA-Land", cer, ody_c, cells_c)]
print("July 2013, 06–06 UTC, Belgian cells of each grid: bridge and ODYSSEY on the same cells (mm)")
for name, b, o, cells in rows:
    bv, ov = b.ravel()[cells], o[cells]
    ok = np.isfinite(bv) & np.isfinite(ov)
    print(f"  {name:10s} {ok.sum():5d} cells | mean {bv[ok].mean():5.1f} vs {ov[ok].mean():6.1f}"
          f" | median {np.median(bv[ok]):5.1f} vs {np.median(ov[ok]):6.1f}"
          f" | ratio of medians {np.median(ov[ok]) / np.median(bv[ok]):.2f}"
          f" | Spearman {spearmanr(bv[ok], ov[ok]).statistic:.2f}")

## 4. Maps and cell-by-cell comparison

Top: ODYSSEY at 2 km, and the two bridges on the same grid (ERA5-Land is land only: no value over the sea).
Bottom: ODYSSEY averaged over the bridge cells, and cell against cell, on log axes, over the whole domain (grey)
and over Belgium (colour). ODYSSEY and the bridges have separate colour scales, because their levels differ.

In [ ]:
p = Proj(where["projdef"])
geom = transform(country_geometry("BEL", str(DATA_DIR / "naturalearth")), lambda c: np.column_stack(p(c[:, 0], c[:, 1])))
x0, y0 = p(float(where["UL_lon"]), float(where["UL_lat"]))
ext = [0, ody.shape[1] * 2, ody.shape[0] * 2, 0]                                # km from the crop corner
outline = [((np.asarray(g.exterior.xy[0]) - x0) / 1e3, (y0 - np.asarray(g.exterior.xy[1])) / 1e3)
           for g in getattr(geom, "geoms", [geom])]

v_ody = np.nanpercentile(ody, 99)
v_br = np.nanpercentile(np.concatenate([era.ravel(), cer.ravel()]), 99)
maps = [("ODYSSEY, 2 km", ody, v_ody), ("ERA5-Land on the OPERA grid", era.ravel()[ie], v_br),
        ("CERRA-Land on the OPERA grid", cer.ravel()[ic], v_br),
        ("ODYSSEY on ERA5-Land cells", ody_e[ie], v_ody), ("ODYSSEY on CERRA-Land cells", ody_c[ic], v_ody)]
fig, axs = plt.subplots(2, 3, figsize=(15, 9.5), constrained_layout=True)
for ax, (title, f, vmax) in zip(axs.flat, maps):
    im = ax.imshow(f, extent=ext, vmin=0, vmax=vmax, cmap="viridis", interpolation="nearest")
    for x, y in outline:
        ax.plot(x, y, color="w", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("km")
    fig.colorbar(im, ax=ax, shrink=0.8, label="mm" if vmax == v_br else "mm (index, own scale)")
fig.suptitle("1/7/2013 06 UTC → 1/8/2013 06 UTC")

ax = axs[1, 2]
for name, b, o, cells in rows:
    ax.scatter(b.ravel(), o, s=2, color="0.8")
for (name, b, o, cells), c, s in zip(rows[::-1], ["C1", "C0"], [4, 10]):
    ax.scatter(b.ravel()[cells], o[cells], s=s, color=c, label=f"{name}, Belgian cells")
ax.set_xscale("log")
ax.set_yscale("log")
ax.axline((10, 10), (100, 100), color="k", lw=0.8)
ax.set_xlabel("bridge (mm)")
ax.set_ylabel("ODYSSEY on the bridge cell (mm)")
ax.set_title("cell against cell (grey: whole domain)")
ax.legend()
fig.savefig(OUT / "bridge_2013-07.png", dpi=110)

## 5. What to take from it

- Both bridges read, cover the domain and land on the OPERA grid as whole cells; ERA5-Land has no value over
  the sea.
- The ODYSSEY level is well above both bridges, as expected from a maximum-reflectivity composite: the level
  gap is what H3 corrects, and the ratios above are one month only.
- The brightest ODYSSEY spots (around the radars and in the south-east of Belgium) survive the cleaning with a
  static mask counted on one month; they belong to H2.